In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error,make_scorer
import time
#import pygwalker as pyg
from datetime import datetime
import os
#from ydata_profiling import ProfileReport
import csv
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV

In [2]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

## Imputing missing values ##

In [3]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [4]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [5]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


## Feature Engineering ##

In [6]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [7]:
df['Assortment'] = np.where(df['Assortment'] == 'b','b','other')

In [8]:
df.sample(5)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,PromoInterval,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,MonthStr,IsPromoMonth
802611,597,5,2013-07-12,4655,675,1,0,0,1,a,...,"Mar,Jun,Sept,Dec",2013,7,12,0,58.0,28,18.75,Jul,0
386762,939,4,2014-07-24,5194,495,1,0,0,1,d,...,NaN,2014,7,24,1,18.0,30,0.00,Jul,0
422960,46,2,2014-06-17,6858,753,1,1,0,1,c,...,"Jan,Apr,Jul,Oct",2014,6,17,0,105.0,25,38.75,Jun,0
947230,266,1,2013-03-04,9037,894,1,1,0,0,a,...,"Mar,Jun,Sept,Dec",2013,3,4,0,0.0,10,24.25,Mar,1
540187,198,2,2014-03-04,5882,583,1,1,0,0,a,...,"Jan,Apr,Jul,Oct",2014,3,4,0,138.0,10,47.25,Mar,0


In [9]:
df = df[df['Open'] == 1].copy()

In [10]:
df = df[df['Date'] != pd.Timestamp('2015-07-04')]

In [11]:
df.shape

(843278, 27)

In [12]:
df = df.sort_values('Date')

cutoff_date = '2015-06-01'

train = df[df['Date'] < cutoff_date]
valid = df[df['Date'] >= cutoff_date]

store_avg_sales = train.groupby('Store')['Sales'].mean().rename('Store_avg_sales')
store_avg_customers = train.groupby('Store')['Customers'].mean().rename('Store_avg_customers')
train = train.merge(store_avg_sales, on='Store', how='left')
train = train.merge(store_avg_customers, on='Store', how='left')
valid = valid.merge(store_avg_sales, on='Store', how='left')
valid = valid.merge(store_avg_customers, on='Store', how='left')

num_col=['CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths','Store_avg_sales','Store_avg_customers']
cat_col = ['Year','StoreType','Assortment']

X_train = train.drop(['Sales', 'Date'], axis=1)
y_train = train['Sales']

X_test = valid.drop(['Sales', 'Date'], axis=1)
y_test = valid['Sales']

In [13]:
X_train.drop(['Open','Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)
X_test.drop(['Open','Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [14]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first',sparse_output=True),cat_col)
],remainder='passthrough')

In [15]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [16]:
y_train_log = np.log1p(y_train)

In [17]:
df = df.sort_values('Date').reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=5)

df = df.merge(store_avg_sales, on='Store', how='left')
df = df.merge(store_avg_customers, on='Store', how='left')

df.drop(columns=['Open','Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'],axis=1,inplace=True)

def rmspe(y_true, y_pred):
    y_true = np.expm1(y_true)   # since we're working in log-space during CV
    y_pred = np.expm1(y_pred)
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

# lower is better, so greater_is_better=False
rmspe_scorer = make_scorer(rmspe, greater_is_better=False)

param_distributions = {
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [300, 500, 800],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

xgb_model = XGBRegressor(tree_method='hist', n_jobs=-1, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=30,              # try 20 random combinations
    scoring=rmspe_scorer,
    cv=tscv,                # time-aware folds, not random
    verbose=2,
    random_state=42,
    n_jobs=1                # keep this 1 since XGBoost itself already uses n_jobs=-1 internally
)
df_sample = df.sample(frac=0.3, random_state=42).sort_values('Date').reset_index(drop=True)

X_full = preprocessor.fit_transform(df_sample.drop(['Sales', 'Date'], axis=1))
y_full_log = np.log1p(df_sample['Sales'])

search.fit(X_full, y_full_log)

print("Best params:", search.best_params_)
print("Best RMSPE:", -search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=10, n_estimators=300, subsample=0.8; total time=   7.7s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=10, n_estimators=300, subsample=0.8; total time=  10.2s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=10, n_estimators=300, subsample=0.8; total time=  11.2s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=10, n_estimators=300, subsample=0.8; total time=  13.8s


KeyboardInterrupt: 

In [18]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    # 'XGBRegressor' : XGBRegressor(subsample= 0.8, n_estimators= 500, max_depth= 8, learning_rate= 0.05, colsample_bytree= 1.0,tree_method='hist',n_jobs=-1,random_state=42),
    "XGBRegressor" : XGBRegressor(
    tree_method='hist', n_jobs=-1, random_state=42,
    max_depth=8, learning_rate=0.05, n_estimators=500,
    subsample=0.8, colsample_bytree=0.8), 
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train_log)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    pred_log = model.predict(X_test)
    prediction_stop = time.perf_counter()

    Prediction = np.expm1(pred_log)
    #prediction = pred_log

    prediction_time_taken = prediction_stop-prediction_start
    # rmse = root_mean_squared_error(y_test,prediction)
    # rmsep = rmse/y_test_mean

    def rmspe(y_true, y_pred):
        mask = y_true != 0
        return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

    score = rmspe(y_test.values, Prediction)


    print(f'{model_name}: {score*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    # importance = pd.Series(model.feature_importances_, index=preprocessor.get_feature_names_out())
    # print(importance.sort_values(ascending=False).head(25))

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        #'rmse': rmse,
        'rmsep_percent': score * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

    errors = np.abs((y_test.values - Prediction) / y_test.values)
    error_df = valid.copy()
    error_df['prediction'] = Prediction
    error_df['pct_error'] = errors

    print(error_df.sort_values('pct_error', ascending=False).head(30)['Store'].value_counts())
    print(error_df.sort_values('pct_error', ascending=False).head(30)['Date'].value_counts())


    print(error_df.sort_values('pct_error', ascending=False).head(30)[
        ['Store','Date','Sales','prediction','pct_error','Promo','StateHoliday','StoreType','DayOfWeek']
    ])

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)



XGBRegressor: 14.17%

Training time taken for XGBRegressor: 44.9874

prediction time taken for XGBRegressor: 0.2778

Store
415     5
292     2
909     2
782     2
876     2
749     2
286     2
917     1
956     1
947     1
501     1
1014    1
526     1
630     1
76      1
139     1
125     1
534     1
607     1
39      1
Name: count, dtype: int64
Date
2015-07-25    4
2015-06-27    3
2015-06-13    3
2015-07-13    2
2015-06-20    2
2015-06-03    1
2015-06-04    1
2015-06-26    1
2015-07-01    1
2015-07-10    1
2015-06-05    1
2015-06-06    1
2015-06-01    1
2015-07-08    1
2015-07-15    1
2015-06-29    1
2015-07-20    1
2015-06-08    1
2015-06-18    1
2015-07-02    1
2015-06-12    1
Name: count, dtype: int64
       Store       Date  Sales    prediction  pct_error  Promo  StateHoliday  \
36352    292 2015-07-10   1012   5503.925781   4.438662      0             0   
29038    909 2015-07-01   3547  14066.365234   2.965708      1             0   
24938    782 2015-06-26   1422   5382.176758